# Piko & Lumi — OmniLite Studio v0.2

**Auto-batch Wan2.2 Image-to-Video**. Jalankan **Runtime → Change runtime type → T4 GPU**, kemudian **Runtime → Run all**. Anda hanya perlu mengunggah **satu gambar referensi adegan 16:9**. Notebook akan menyiapkan lingkungan, membuat klip secara berurutan, melanjutkan dari frame terakhir, menggabungkan video, dan mengunduh ZIP.

### Apa yang otomatis?
1. Menyiapkan engine Wan2.2 + LightX2V dari [notebook komunitas Collab-AI-Gen](https://github.com/Collab-AI-Gen/Google-Colab_Notebooks/blob/main/wan2_2/wan22_Lightx2v_Aigen.ipynb), dengan beberapa penyesuaian dependensi untuk Colab.
2. Meminta unggahan **satu** gambar adegan Piko dan Lumi (bukan character sheet).
3. Menjalankan 1–3 prompt adegan dan menyimpan MP4 setiap adegan.
4. Menggunakan frame terakhir klip sebelumnya sebagai gambar referensi berikutnya.
5. Menggabungkan klip menjadi video pendek 1080p (resizing biasa), membuat ZIP dan memulai download.

**Batasan:** Ini bukan clone model Google Omni Flash dan bukan editor video multimodal native. Kualitas sambungan tidak dijamin. GPU gratis mungkin tidak tersedia dan bisa terputus. Simpan ZIP segera. Jangan menjalankan kode yang Anda tidak percayai. Engine komunitas dijalankan dari sumber eksternal dan versinya terkunci lewat checksum untuk mencegah perubahan tak terduga. Notebook ini belum diuji end-to-end pada GPU T4 oleh pembuatnya.

### Tutorial cepat
- Pilih **T4 GPU**, lalu pilih **Run all**.
- Saat diminta, unggah gambar Piko & Lumi berlatar taman pelangi.
- Tunggu batch berjalan dalam sesi Colab aktif. Setelah selesai, ZIP akan diunduh.
- **Untuk produksi banyak video, buat gambar referensi baru dan ubah 3 prompt di sel konfigurasi.**


In [ ]:
# @title 1. Periksa T4, instal engine Wan2.2, dan perbaiki dependensi
import os, sys, json, hashlib, urllib.request, time

import torch
assert torch.cuda.is_available(), (
    "CUDA tidak aktif. Di Google Colab pilih Runtime → Change runtime type → T4 GPU, lalu jalankan ulang."
)
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)

# Mengambil notebook komunitas yang sebelumnya dipakai, tetapi tidak menerbitkan ulang isinya.
UPSTREAM_URL = ("https://raw.githubusercontent.com/Collab-AI-Gen/"
                "Google-Colab_Notebooks/main/wan2_2/wan22_Lightx2v_Aigen.ipynb")
EXPECTED_GIT_BLOB_SHA1 = "3f3903fcb23d59bbf5789633611e9ea341be7aec"
raw = urllib.request.urlopen(UPSTREAM_URL, timeout=90).read()
git_blob_sha1 = hashlib.sha1(
    f"blob {len(raw)}\\0".encode().replace(b"\\0", b"\\x00") + raw
).hexdigest()
if git_blob_sha1 != EXPECTED_GIT_BLOB_SHA1:
    raise RuntimeError(
        "Notebook upstream berubah sejak versi yang diperiksa. "
        "Demi keamanan, setup dihentikan; minta versi OmniLite terbaru."
    )
upstream = json.loads(raw.decode("utf-8"))
setup_source = upstream["cells"][1]["source"]
if isinstance(setup_source, list):
    setup_source = "".join(setup_source)

# Hindari pin torch/xformers/SageAttention lama yang sering gagal pada Python Colab baru.
old_install = "!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 triton==3.2.0 sageattention==1.0.6"
assert old_install in setup_source, "Struktur notebook upstream berubah; setup otomatis dihentikan."
setup_source = setup_source.replace(
    "!pip install torch==2.6.0 torchvision==0.21.0",
    "print('Memakai versi PyTorch bawaan Colab untuk menghindari pin versi lama.')"
)
setup_source = setup_source.replace(
    old_install,
    "!pip install -q torchsde einops diffusers accelerate"
)
setup_source = setup_source.replace(
    "!pip install av spandrel albumentations insightface onnx opencv-python segment_anything ultralytics onnxruntime",
    "!pip install -q av spandrel albumentations onnx opencv-python segment_anything ultralytics onnxruntime"
)
setup_source = setup_source.replace(
    "!pip install onnxruntime-gpu -y",
    "# onnxruntime-gpu sengaja dilewati; tidak diperlukan untuk batch I2V"
)
setup_source = setup_source.replace(
    "!git clone --branch ComfyUI_v0.3.47 https://github.com/Isi-dev/ComfyUI",
    "!test -d /content/ComfyUI/.git || git clone --branch ComfyUI_v0.3.47 https://github.com/Isi-dev/ComfyUI /content/ComfyUI"
)
setup_source = setup_source.replace(
    "!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git",
    "!test -d /content/ComfyUI/custom_nodes/ComfyUI_GGUF/.git || git clone https://github.com/Isi-dev/ComfyUI_GGUF.git"
)
setup_source = setup_source.replace(
    "!git clone --branch kjnv1.1.3 https://github.com/Isi-dev/ComfyUI_KJNodes.git",
    "!test -d /content/ComfyUI/custom_nodes/ComfyUI_KJNodes/.git || git clone --branch kjnv1.1.3 https://github.com/Isi-dev/ComfyUI_KJNodes.git"
)
setup_source = setup_source.replace(
    "!git clone https://github.com/Isi-dev/Practical-RIFE",
    "!test -d /content/Practical-RIFE/.git || git clone https://github.com/Isi-dev/Practical-RIFE /content/Practical-RIFE"
)

print("Menjalankan Prepare Environment; download model pertama kali bisa besar.")
result = get_ipython().run_cell(setup_source, store_history=False)
if result.error_before_exec or result.error_in_exec:
    raise RuntimeError("Setup Wan2.2 gagal; periksa output sel ini.") from (
        result.error_in_exec or result.error_before_exec
    )

assert "generate_video" in globals(), "Fungsi generate_video tidak berhasil disiapkan."
assert "upload_image" in globals(), "Fungsi upload_image tidak berhasil disiapkan."
import torchsde
print("✅ Engine Wan2.2 & torchsde siap. SageAttention tidak akan dipakai.")


In [ ]:
# @title 2. Unggah satu gambar referensi 16:9
from pathlib import Path
from IPython.display import display
from PIL import Image as PILImage

file_uploaded = upload_image()
assert file_uploaded and Path(file_uploaded).exists(), "Gambar belum berhasil diunggah."
ref_image = PILImage.open(file_uploaded)
print(f"✅ Referensi: {file_uploaded} | {ref_image.width} x {ref_image.height}")
display(ref_image.copy().resize((640, 360)))


In [ ]:
# @title 3. Pengaturan episode (bisa diubah lewat formulir Colab)
# @markdown Mulai dengan 1 adegan bila ingin pengujian singkat; pilih 3 untuk batch utuh.
number_of_scenes = 3 # @param {type:"integer"}
save_to_drive = False # @param {type:"boolean"}
regenerate_existing = False # @param {type:"boolean"}
render_width = 832 # @param {type:"integer"}
render_height = 480 # @param {type:"integer"}
output_fps = 30 # @param {type:"integer"}
final_duration_seconds = 8 # @param {type:"integer"}

scene_1_prompt = "Piko the adorable orange fox wearing a turquoise scarf and Lumi the adorable purple bunny with a yellow star on its chest dance happily side by side in a vibrant rainbow playground. Full body, fixed wide shot, playful gentle bouncing, waving arms, detailed joyful 3D preschool animation, original designs remain unchanged." # @param {type:"string"}
scene_2_prompt = "Piko and Lumi dance side to side together with gentle rhythmic steps and expressive smiling faces. Preserve the exact fox scarf, bunny star, colors, face shapes, and magical rainbow playground. Fixed camera, bright 3D preschool animation." # @param {type:"string"}
scene_3_prompt = "Piko and Lumi finish the dance by raising their arms and waving to the camera, with natural ear and scarf motion, standing side by side in the same magical rainbow playground. Preserve all original character details and lighting, joyful 3D children's animation." # @param {type:"string"}

assert 1 <= number_of_scenes <= 3, "number_of_scenes harus 1, 2, atau 3."
assert render_width % 16 == 0 and render_height % 16 == 0, "Resolusi harus kelipatan 16."
print(f"✅ Siap membuat {number_of_scenes} scene @ {render_width}x{render_height}.")


In [ ]:
# @title 4. Render batch otomatis (jangan tutup sesi Colab)
import os, gc, cv2, json, time, shutil, subprocess, zipfile
from pathlib import Path

from google.colab import files

if save_to_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        studio = Path("/content/drive/MyDrive/Piko_Lumi_OmniLite")
    except Exception as e:
        print("Google Drive gagal terhubung; menggunakan penyimpanan lokal:", str(e))
        studio = Path("/content/Piko_Lumi_OmniLite")
else:
    studio = Path("/content/Piko_Lumi_OmniLite")

studio.mkdir(parents=True, exist_ok=True)
reference = Path(file_uploaded)
assert reference.is_file(), "Referensi tidak ditemukan; jalankan sel Upload Image."
assert torch.cuda.is_available(), "GPU tidak terhubung."
assert callable(generate_video), "Engine generator belum siap."

scene_defs = [
    ("01_Happy_Dance", scene_1_prompt),
    ("02_Side_to_Side", scene_2_prompt),
    ("03_Happy_Ending", scene_3_prompt),
][:number_of_scenes]
negative = ("low quality, character morphing, changing clothes, duplicate characters, "
            "extra limbs, deformed face, unnatural motion, strong camera motion, "
            "flickering, unexpected text, subtitles, watermarks")

def read_last_frame(mp4: Path, dest: Path) -> Path:
    cap = cv2.VideoCapture(str(mp4))
    if not cap.isOpened():
        raise RuntimeError(f"MP4 tidak bisa dibaca: {mp4}")
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(count - 1, 0))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Frame terakhir gagal dibaca: {mp4}")
    if not cv2.imwrite(str(dest), frame):
        raise RuntimeError(f"Gagal menyimpan referensi frame: {dest}")
    return dest

def verify_mp4(p: Path) -> None:
    if not p.is_file() or p.stat().st_size < 1000:
        raise RuntimeError(f"Video kosong atau tidak tersedia: {p}")
    cap = cv2.VideoCapture(str(p))
    valid = cap.isOpened() and int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) > 0
    cap.release()
    if not valid:
        raise RuntimeError(f"Video tidak valid: {p}")

shutil.copy2(reference, studio / ("original_reference" + reference.suffix))
ref_for_next = reference
completed = []
manifest = {"engine": "Wan2.2 community LightX2V", "version": "OmniLite-v0.2",
            "scene_config": [], "render_width": render_width,
            "render_height": render_height, "fps_original": 16}

for index, (name, instruction) in enumerate(scene_defs, 1):
    target = studio / f"{name}.mp4"
    snapshot = studio / f"{name}_last_frame.png"
    print(f"\\n▶ {index}/{len(scene_defs)}: {name}", flush=True)

    if target.exists() and not regenerate_existing:
        verify_mp4(target)
        print("   Melewati file yang sudah ada.")
    else:
        generated = Path("/content/ComfyUI/output/ComfyUI.mp4")
        generated.unlink(missing_ok=True)  # mencegah menyalin video lama saat render gagal
        t0 = time.time()
        generate_video(
            image_path=str(ref_for_next),
            positive_prompt=instruction,
            prompt_assist="none",
            negative_prompt=negative,
            width=render_width,
            height=render_height,
            frames=49,
            fps=16,
            seed=42+index,
            steps=6,
            cfg_scale=1,
            sampler_name="euler",
            scheduler="simple",
            output_format="mp4",
            overwrite=True,
            use_sage_attention=False,
            use_lightx2v=True,
            lightx2v_Strength=3,
            lightx2v_steps=6,
            use_pusa=True,
            pusa_Strength=1.5,
            pusa_steps=6,
            enable_flow_shift=True,
            shift=8,
            enable_flow_shift2=True,
            shift2=8,
            end_step1=3,
            use_lora=False,
            use_lora2=False,
            use_lora3=False,
            rel_l1_thresh=0,
        )
        verify_mp4(generated)
        shutil.copy2(generated, target)
        print(f"   ✅ Selesai dalam {(time.time()-t0)/60:.1f} menit:", target.name)

    completed.append(target)
    ref_for_next = read_last_frame(target, snapshot)
    manifest["scene_config"].append({
        "scene": index, "name": name, "prompt": instruction, "file": target.name
    })
    (studio/"manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Semua adegan yang diminta berhasil diproses.")


In [ ]:
# @title 5. Gabungkan ke MP4, buat ZIP, dan download otomatis
from pathlib import Path
from google.colab import files
import subprocess, shutil

assert "completed" in globals() and completed, "Jalankan sel render batch terlebih dahulu."
playlist = studio / "clips_to_concat.txt"
playlist.write_text(
    "".join(f"file '{str(p)}'\n" for p in completed),
    encoding="utf-8"
)
final_video = studio / "Piko_Lumi_OmniLite_1080p.mp4"

cmd = [
    "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(playlist),
    "-t", str(final_duration_seconds),
    "-vf",
    f"scale=1920:1080:force_original_aspect_ratio=decrease:flags=lanczos,"
    f"pad=1920:1080:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={output_fps}",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
    "-pix_fmt", "yuv420p", "-an", str(final_video)
]
subprocess.run(cmd, check=True)
assert final_video.exists() and final_video.stat().st_size > 0

# ZIP disimpan di luar folder untuk menghindari zip-di-dalam-zip.
archive = shutil.make_archive("/content/Piko_Lumi_OmniLite_v02", "zip", studio)
print("\n✅ MP4:", final_video)
print("✅ ZIP:", archive)
print("Catatan: output 1080p berasal dari resize; bukan detail 1080p native atau RIFE.")
print("Jangan tutup Colab sampai unduhan selesai.")
files.download(archive)
